# ПЗ3. Потокова обробка: Apache Kafka + Spark Structured Streaming
Датасет: Bitcoin Historical Data (1-хвилинні свічки BTC/USD). Записник виконує **всі завдання 1–8** методички у порядку методички.
Єдиний ручний крок – підключення Google Drive. Далі: **Runtime → Run all** (орієнтовно 20–25 хв).

| Завдання | Розділ |
|---|---|
| 1. Підготовка Kafka, topic `bigdata-topic` | §2 |
| 2. Producer із затримкою | §3 |
| 3–5. Spark Structured Streaming: отримання, обробка, вікна | §4–§5 |
| 6. Виявлення аномалій | §4 (потік) і §7 (калібрування k на всій історії) |
| 7. Збереження результатів (parquet/csv + checkpoint) | §5–§6 |
| 8. Аналіз закономірностей | §8 |

Скріншоти робить сам записник: вивід реальних команд у вигляді термінального PNG і Spark UI через headless Chromium. Усе зберігається у `MyDrive/АВД/colab_results/pz3/`.

## 0. Підключення Drive, налаштування, допоміжні функції

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys, subprocess, json, time, glob, shutil, textwrap, re, platform
from contextlib import contextmanager
import numpy as np, pandas as pd
from PIL import Image, ImageDraw, ImageFont

# ---------------- параметри (можна змінювати) ----------------
TOPIC          = "bigdata-topic"
# періоди відтворення (UTC, межі включно): COVID-крах, травневий крах 2021, колапс FTX
PERIODS        = "2020-03-09:2020-03-16,2021-05-17:2021-05-22,2022-11-06:2022-11-12"
DELAY_MS       = 10          # затримка між повідомленнями producer, мс
WATERMARK      = "10 minutes"
TRIGGER_SEC    = 10
K_STREAM       = 5           # початкове k у потоці; обране за даними k визначається у §7
KAFKA_VERSION  = "3.9.1"; SCALA = "2.13"
PYSPARK_VER    = "3.5.3"
DRIVE_ROOT     = "/content/drive/MyDrive/АВД"
OUT_DRIVE      = f"{DRIVE_ROOT}/colab_results/pz3"
# --------------------------------------------------------------
BASE="/content/pz3"; CODE=f"{BASE}/code"; DATA=f"{BASE}/data"; LOGS=f"{BASE}/logs"
SHOTS=f"{BASE}/screenshots"; RES=f"{BASE}/results"; JOB_OUT=f"{BASE}/job_out"; CKPT=f"{BASE}/ckpt"
FLAG=f"{BASE}/producer_done.json"
for d in (CODE, DATA, LOGS, SHOTS, RES): os.makedirs(d, exist_ok=True)
BOOT="localhost:9092"
CHECKS = {}      # завдання -> (True/False, повідомлення)
TIMES  = []      # (етап, секунди)
SHOT_LIST = []

@contextmanager
def timed(name):
    t=time.time(); print(f"--> {name}")
    try: yield
    finally:
        TIMES.append((name, round(time.time()-t,1))); print(f"<-- {name}: {time.time()-t:.1f} с")

def sync_drive():
    """Копіює результати на Drive (викликається кілька разів, щоб частковий результат не втрачався)."""
    try:
        os.makedirs(OUT_DRIVE, exist_ok=True)
        for sub in ("screenshots","results","logs"):
            if os.path.isdir(f"{BASE}/{sub}"):
                shutil.copytree(f"{BASE}/{sub}", f"{OUT_DRIVE}/{sub}", dirs_exist_ok=True)
        if os.path.isdir(JOB_OUT):
            shutil.copytree(JOB_OUT, f"{OUT_DRIVE}/results/job_out", dirs_exist_ok=True,
                            ignore=shutil.ignore_patterns("scored_parts"))
        for f in ("versions.txt","runtimes.csv","summary.txt"):
            if os.path.exists(f"{BASE}/{f}"): shutil.copy(f"{BASE}/{f}", f"{OUT_DRIVE}/{f}")
    except Exception as e:
        print("УВАГА: не вдалося синхронізувати з Drive:", e)

def write_summary():
    lines=["ПЗ3 – чекліст завдань", "="*50]
    for k,(ok,msg) in CHECKS.items(): lines.append(f"[{'OK' if ok else 'FAIL'}] {k}: {msg}")
    open(f"{BASE}/summary.txt","w",encoding="utf-8").write("\n".join(lines)+"\n")
    pd.DataFrame(TIMES, columns=["stage","seconds"]).to_csv(f"{BASE}/runtimes.csv", index=False)
    sync_drive()
    return "\n".join(lines)

def check(name, ok, msg=""):
    """М'яка перевірка: фіксує OK/FAIL і продовжує."""
    CHECKS[name]=(bool(ok), msg); print(("[OK]   " if ok else "[FAIL] ")+name+(": "+msg if msg else ""))

def require(ok, msg):
    """Критична перевірка: при невдачі зберігає звіт і зупиняє виконання з чітким повідомленням."""
    if not ok:
        CHECKS["КРИТИЧНА ПОМИЛКА"]=(False,msg); print(write_summary())
        raise RuntimeError("ПЗ3 ЗУПИНЕНО: "+msg)

# ---- термінальний PNG зі справжнім виводом команди ----
ANSI = re.compile(r"\x1b\[[0-9;?]*[A-Za-z]")
def _font(sz=15):
    for p in ("/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf","/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf"):
        if os.path.exists(p): return ImageFont.truetype(p, sz)
    return ImageFont.load_default()
def term_png(cmd, text, path, width=118, max_lines=70):
    text = ANSI.sub("", text).replace("\t","    ")
    lines=[]
    for ln in text.splitlines():
        lines += textwrap.wrap(ln, width, drop_whitespace=False, replace_whitespace=False) or [""]
    if len(lines)>max_lines:
        cut=len(lines)-max_lines
        lines = lines[:max_lines//2] + [f"[... пропущено {cut} рядків для скріншота; повний вивід у logs/ ...]"] + lines[-max_lines//2:]
    cl = textwrap.wrap("$ "+cmd, width, subsequent_indent="  ", break_long_words=True) or ["$"]
    f=_font(15); cw=int(f.getlength("M")); lh=21; pad=14
    img=Image.new("RGB",(cw*width+2*pad, lh*(len(cl)+len(lines)+1)+2*pad),(30,30,30))
    dr=ImageDraw.Draw(img); y=pad
    for l in cl: dr.text((pad,y),l,font=f,fill=(120,220,120)); y+=lh
    for l in lines: dr.text((pad,y),l,font=f,fill=(212,212,212)); y+=lh
    img.save(path)

def sh(cmd, shot=None, timeout=300, env=None, quiet=False, log=None):
    """Виконує команду в bash. Повертає (код, вивід). shot='ім'я' -> PNG з реальним виводом."""
    e=os.environ.copy(); 
    if env: e.update(env)
    try:
        p=subprocess.run(cmd, shell=True, executable="/bin/bash", capture_output=True, text=True, timeout=timeout, env=e)
        out=(p.stdout or "")+(p.stderr or ""); rc=p.returncode
    except subprocess.TimeoutExpired as ex:
        out=(ex.stdout.decode() if isinstance(ex.stdout,bytes) else (ex.stdout or ""))+"\n[TIMEOUT]"; rc=124
    if not quiet: print(out[-3000:] if len(out)>3000 else out)
    if log: open(f"{LOGS}/{log}","w",encoding="utf-8").write(f"$ {cmd}\n{out}")
    if shot:
        try:
            term_png(cmd, out, f"{SHOTS}/{shot}.png"); SHOT_LIST.append(shot)
        except Exception as ex: print("скріншот не створено:", ex)
    return rc, out

t_global=time.time()
print("Готово. Результати:", OUT_DRIVE)

## 1. Дані, Java, Kafka (KRaft), PySpark

In [ ]:
# --- пошук і розпакування архіву з датасетом
with timed("Дані: пошук і розпакування archive.zip"):
    zip_path = f"{DRIVE_ROOT}/archive.zip"
    if not os.path.exists(zip_path):
        found = glob.glob("/content/drive/MyDrive/**/archive.zip", recursive=True)
        require(found, "archive.zip не знайдено на Drive: покладіть файл у MyDrive/АВД/archive.zip")
        zip_path = found[0]
    print("архів:", zip_path)
    sh(f"rm -rf {DATA}/* && unzip -o -q '{zip_path}' -d {DATA}", quiet=True)
    csvs = sorted(glob.glob(f"{DATA}/**/*.csv", recursive=True), key=os.path.getsize, reverse=True)
    require(csvs, "у архіві немає csv-файлу")
    CSV = csvs[0]
    print("CSV:", CSV, f"{os.path.getsize(CSV)/1e6:.0f} МБ")
    ts_all = pd.read_csv(CSV, usecols=["Timestamp"])["Timestamp"].astype("int64")
    print("рядків:", len(ts_all), "| від", pd.to_datetime(ts_all.min(),unit="s",utc=True), "до", pd.to_datetime(ts_all.max(),unit="s",utc=True))
    EXPECTED = {}
    for it in PERIODS.split(","):
        s,e = it.split(":")
        lo=int(pd.Timestamp(s,tz="UTC").timestamp()); hi=int((pd.Timestamp(e,tz="UTC")+pd.Timedelta(days=1)).timestamp())
        EXPECTED[it]=int(((ts_all>=lo)&(ts_all<hi)).sum())
    EXPECTED_TOTAL=sum(EXPECTED.values())
    print("рядків у періодах відтворення:", EXPECTED, "| разом", EXPECTED_TOTAL)
    pd.DataFrame([{"period":k,"rows_in_csv":v} for k,v in EXPECTED.items()]).to_csv(f"{RES}/replay_periods.csv", index=False)
    del ts_all
    require(EXPECTED_TOTAL>0, "жоден із періодів відтворення не містить даних")

In [ ]:
# --- Java 17
with timed("Встановлення OpenJDK 17"):
    sh("apt-get -qq update && apt-get -qq install -y openjdk-17-jdk-headless", quiet=True, timeout=600)
    jh = sorted(glob.glob("/usr/lib/jvm/java-17-openjdk*"))
    require(jh, "OpenJDK 17 не встановлено (apt-get)")
    os.environ["JAVA_HOME"]=jh[0]; os.environ["PATH"]=jh[0]+"/bin:"+os.environ["PATH"]
    print(sh("java -version", quiet=True)[1])

In [ ]:
# --- Kafka 3.9.x (режим KRaft, без ZooKeeper)
with timed("Завантаження Kafka"):
    KAFKA=None
    for ver_ in (KAFKA_VERSION, "3.9.0", "3.8.1"):          # запасні версії, якщо основної немає на дзеркалах
        tgz=f"kafka_{SCALA}-{ver_}.tgz"; cand=f"/content/kafka_{SCALA}-{ver_}"
        if os.path.isdir(cand): KAFKA=cand; break
        for base in ("https://dlcdn.apache.org/kafka","https://archive.apache.org/dist/kafka"):
            url=f"{base}/{ver_}/{tgz}"; print("спроба:", url)
            sh(f"wget -q -O /content/{tgz} {url}", quiet=True, timeout=600)
            if os.path.exists(f"/content/{tgz}") and os.path.getsize(f"/content/{tgz}")>50e6:
                sh(f"tar -xzf /content/{tgz} -C /content", quiet=True); KAFKA=cand; break
        if KAFKA: break
    require(KAFKA, "не вдалося завантажити Kafka ні з dlcdn.apache.org, ні з archive.apache.org")
    require(os.path.isdir(KAFKA), "Kafka не розпаковано")
    print("Kafka:", KAFKA)

In [ ]:
# --- pip-пакети: pyspark (версія = версії пакета spark-sql-kafka), confluent-kafka, playwright
with timed("pip: pyspark, confluent-kafka, playwright"):
    sh(f"pip install -q pyspark=={PYSPARK_VER} confluent-kafka pyarrow pillow playwright", quiet=True, timeout=900)
    import importlib.metadata as im
    SPARK_V = im.version("pyspark")
    KAFKA_PKG = f"org.apache.spark:spark-sql-kafka-0-10_2.12:{SPARK_V}"
    print("pyspark", SPARK_V, "| пакет:", KAFKA_PKG, "| confluent-kafka", im.version("confluent-kafka"))
    UI_OK = False
    rc,_ = sh("playwright install --with-deps chromium", quiet=True, timeout=900)
    UI_OK = (rc==0); print("playwright/chromium встановлено:", UI_OK)

In [ ]:
# --- ініціалізація сховища KRaft і запуск брокера у фоні
with timed("Запуск Kafka (KRaft)"):
    cfg = f"{KAFKA}/config/kraft/server.properties"
    if not os.path.exists(cfg): cfg = f"{KAFKA}/config/server.properties"
    sh(f"rm -rf /content/kafka-logs; sed -i 's#^log.dirs=.*#log.dirs=/content/kafka-logs#' {cfg}", quiet=True)
    cid = sh(f"{KAFKA}/bin/kafka-storage.sh random-uuid", quiet=True)[1].strip().splitlines()[-1]
    sh(f"{KAFKA}/bin/kafka-storage.sh format -t {cid} -c {cfg}",
       shot="01_kafka_storage_format", log="kafka_format.log")
    kenv = os.environ.copy(); kenv["KAFKA_HEAP_OPTS"]="-Xmx1g -Xms1g"
    klog = open(f"{LOGS}/kafka_server.log","w")
    KPROC = subprocess.Popen([f"{KAFKA}/bin/kafka-server-start.sh", cfg], stdout=klog, stderr=subprocess.STDOUT, env=kenv)
    up=False
    for i in range(45):
        time.sleep(2)
        if KPROC.poll() is not None: break
        rc,_ = sh(f"{KAFKA}/bin/kafka-broker-api-versions.sh --bootstrap-server {BOOT}", quiet=True, timeout=30)
        if rc==0: up=True; break
    require(up, "брокер Kafka не запустився за 90 с. Кінець логу:\n"+sh(f"tail -n 25 {LOGS}/kafka_server.log", quiet=True)[1])
    print("Брокер Kafka працює (KRaft, один вузол: broker+controller).")

In [ ]:
# --- версії та конфігурація машини
def _o(c): return sh(c, quiet=True)[1].strip()
ver = [
 "Python: "+platform.python_version(),
 "Java: "+_o("java -version 2>&1 | head -n1"),
 "Kafka: "+_o(f"{KAFKA}/bin/kafka-topics.sh --version | head -n1")+f" (Scala {SCALA}, режим KRaft)",
 "PySpark: "+SPARK_V, "Пакет Kafka-конектора: "+KAFKA_PKG,
 "confluent-kafka: "+im.version("confluent-kafka"), "pandas: "+pd.__version__, "numpy: "+np.__version__,
 "CPU: "+_o("lscpu | grep 'Model name' | sed 's/.*: *//'")+f" | потоків: {os.cpu_count()}",
 "RAM: "+_o("grep MemTotal /proc/meminfo"), "Дата запуску: "+time.strftime("%Y-%m-%d %H:%M:%S UTC", time.gmtime()),
 f"Параметри: TOPIC={TOPIC}; PERIODS={PERIODS}; DELAY_MS={DELAY_MS}; WATERMARK={WATERMARK}; TRIGGER={TRIGGER_SEC}s"]
open(f"{BASE}/versions.txt","w",encoding="utf-8").write("\n".join(ver)+"\n")
print("\n".join(ver)); sync_drive()

## 2. Завдання 1. Підготовка Kafka та topic `bigdata-topic`
Кількість partition = **1**. Обґрунтування: (1) Kafka гарантує порядок лише всередині partition, а аналіз за часом події й правило аномалій (ковзне вікно 60 хв) потребують хронологічного порядку; (2) при ключі = дата повідомлення різних діб потрапили б у різні partition, і Spark читав би їх паралельно – доба, що відстає, виходила б за 10-хвилинний watermark та відкидалася б як запізніла; (3) навантаження ~100 повідомлень/с на порядки менше за пропускну здатність одного partition; (4) брокер один, тому replication-factor = 1.

In [ ]:
KT = f"{KAFKA}/bin/kafka-topics.sh --bootstrap-server {BOOT}"
with timed("Завдання 1: створення topic"):
    sh(f"{KT} --delete --topic {TOPIC} --if-exists; sleep 2", quiet=True)
    sh(f"{KT} --create --topic {TOPIC} --partitions 1 --replication-factor 1", shot="02_topic_create", log="topic_create.log")
    sh(f"{KT} --list", shot="03_topic_list")
    rc,desc = sh(f"{KT} --describe --topic {TOPIC}", shot="04_topic_describe", log="topic_describe.log")
    check("Завдання 1: topic створено", rc==0 and TOPIC in desc and "PartitionCount: 1" in desc, "kafka-topics --describe")
    require(TOPIC in desc, f"topic {TOPIC} не створено; вивід: {desc}")

## 3. Завдання 2. Producer
Producer (`producer.py`, confluent-kafka) читає CSV, вибирає свічки заданих періодів і послідовно надсилає їх у Kafka: **1 повідомлення = 1 ринкова хвилина**, `key` = дата UTC, `value` = JSON `{ts, open, high, low, close, volume}`. Затримка між повідомленнями (`DELAY_MS`) імітує потік (прискорення ≈ 6000× відносно реального часу при 10 мс). Періоди – волатильні епізоди, щоб відтворення було коротким (≈30 тис. повідомлень).

In [ ]:
%%writefile /content/pz3/code/anomaly_rule.py
# -*- coding: utf-8 -*-
"""Правило виявлення аномалій для потоку 1-хвилинних свічок BTC/USD (ПЗ3, завдання 6).

Формальне правило. Нехай C_t - ціна закриття хвилини t, V_t - обсяг,
r_t = ln(C_t / C_{t-1}) - логарифмічна дохідність.
Базове вікно W_t = {t-60, ..., t-1} (60 попередніх хвилин, поточна не входить).
    sigma_t = max( 1.4826 * MAD_60(r), SIGMA_FLOOR ),   MAD_60(r) = median(|r_i - median(r)|), i in W_t
    z_t     = |r_t| / sigma_t
    q_t     = V_t / median_60(V),  лише якщо median_60(V) > 0
Хвилина t позначається аномальною, якщо  z_t > k  АБО  q_t > k_v.
Вікно вважається коректним, якщо 61 останній запис вкладається у MAX_SPAN секунд
(немає великих пропусків у даних, тому дохідності вікна скінченні й суміжні).
Одна й та сама функція score() використовується і в потоці (foreachBatch), і офлайн.
"""
import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

WINDOW = 60            # довжина ковзного вікна, хв
K_GRID = (3, 5, 7, 10)  # сітка порогів k (k_v береться рівним k)
MAD_SCALE = 1.4826     # переведення MAD у оцінку sigma для нормального розподілу
SIGMA_FLOOR = 1e-4     # мінімальна sigma (1 б.п.): захист від MAD=0 на пласкому ринку
MAX_SPAN = 3900        # макс. тривалість 61 запису, с (60 хв + 5 хв допуску)


def _rolling_median_mad(a, w=WINDOW, chunk=200_000):
    """Для кожного i>=w: медіана та MAD по a[i-w:i]. Обробка блоками, щоб не займати багато RAM."""
    n = len(a)
    med = np.full(n, np.nan)
    mad = np.full(n, np.nan)
    if n <= w:
        return med, mad
    win = sliding_window_view(a, w)      # рядок j = a[j:j+w], вікно перед індексом j+w
    m = n - w
    for s in range(0, m, chunk):
        e = min(s + chunk, m)
        blk = win[s:e]
        md = np.median(blk, axis=1)
        med[s + w:e + w] = md
        mad[s + w:e + w] = np.median(np.abs(blk - md[:, None]), axis=1)
    return med, mad


def score(df, k_grid=K_GRID, w=WINDOW):
    """df: колонки ts (unix с), close, volume. Повертає DataFrame з z_ret, vol_ratio та прапорцями flag_k{k}."""
    d = df[["ts", "close", "volume"]].drop_duplicates("ts").sort_values("ts").reset_index(drop=True)
    n = len(d)
    ts = d["ts"].to_numpy(np.int64)
    c = d["close"].to_numpy(float)
    v = d["volume"].to_numpy(float)
    r = np.full(n, np.nan)
    if n > 1:
        r[1:] = np.diff(np.log(c))
    _, mad_r = _rolling_median_mad(r, w)
    med_v, _ = _rolling_median_mad(v, w)
    sigma = np.maximum(MAD_SCALE * mad_r, SIGMA_FLOOR)
    span = np.full(n, np.inf)
    if n > w + 1:
        span[w + 1:] = ts[w + 1:] - ts[:-(w + 1)]
    valid = (span <= MAX_SPAN) & np.isfinite(r) & np.isfinite(mad_r) & np.isfinite(med_v)
    z = np.where(valid, np.abs(r) / sigma, np.nan)
    with np.errstate(divide="ignore", invalid="ignore"):
        q = np.where(valid & (med_v > 0), v / med_v, np.nan)
    out = d.copy()
    out["r"] = r
    out["sigma"] = np.where(valid, sigma, np.nan)
    out["med_vol"] = np.where(valid, med_v, np.nan)
    out["z_ret"] = z
    out["vol_ratio"] = q
    out["scored"] = valid
    for k in k_grid:
        out[f"flag_k{k}"] = np.nan_to_num(z, nan=0.0) > k
        out[f"flag_k{k}"] |= np.nan_to_num(q, nan=0.0) > k
    return out


class StreamScorer:
    """Стан для foreachBatch: буфер останніх w+1 записів між мікробатчами."""

    def __init__(self, w=WINDOW, k_grid=K_GRID):
        self.w, self.k_grid = w, k_grid
        self.buf = pd.DataFrame(columns=["ts", "close", "volume"])

    def update(self, batch):
        """batch: DataFrame ts, close, volume (лише коректні записи). Повертає оцінки для нових рядків."""
        batch = batch[["ts", "close", "volume"]].drop_duplicates("ts")
        # при повторній доставці (at-least-once) відкидаємо вже оброблені ts
        if len(self.buf):
            batch = batch[~batch["ts"].isin(self.buf["ts"])]
        if batch.empty:
            return batch.assign(r=[], sigma=[], med_vol=[], z_ret=[], vol_ratio=[], scored=[])
        full = pd.concat([self.buf, batch], ignore_index=True)
        sc = score(full, self.k_grid, self.w)
        new_ts = set(batch["ts"].tolist())
        res = sc[sc["ts"].isin(new_ts)].reset_index(drop=True)
        self.buf = full.sort_values("ts").tail(self.w + 1)[["ts", "close", "volume"]].reset_index(drop=True)
        return res

In [ ]:
%%writefile /content/pz3/code/producer.py
# -*- coding: utf-8 -*-
"""Producer: відтворює 1-хвилинні свічки BTC/USD у Kafka (ПЗ3, завдання 2).
1 повідомлення = 1 ринкова хвилина; key = дата UTC (YYYY-MM-DD); value = JSON.
Затримка між повідомленнями імітує потік (прискорене відтворення)."""
import argparse, json, os, sys, time
import pandas as pd
from confluent_kafka import Producer

p = argparse.ArgumentParser()
p.add_argument("--csv", required=True)
p.add_argument("--bootstrap", default="localhost:9092")
p.add_argument("--topic", default="bigdata-topic")
# періоди відтворення: "YYYY-MM-DD:YYYY-MM-DD,..." (межі включно, UTC)
p.add_argument("--periods", default="2020-03-09:2020-03-16,2021-05-17:2021-05-22,2022-11-06:2022-11-12")
p.add_argument("--delay-ms", type=float, default=10.0)
p.add_argument("--max-rows", type=int, default=0, help="0 = без обмеження (для smoke-тесту)")
p.add_argument("--stats", default="")
p.add_argument("--done-flag", default="")
a = p.parse_args()

# --- вибір рядків потрібних періодів (читаємо CSV блоками, щоб не тримати всю історію в RAM)
bounds = []
for item in a.periods.split(","):
    s, e = item.split(":")
    lo = int(pd.Timestamp(s, tz="UTC").timestamp())
    hi = int((pd.Timestamp(e, tz="UTC") + pd.Timedelta(days=1)).timestamp())  # межа виключно
    bounds.append((lo, hi))
parts = []
for ch in pd.read_csv(a.csv, chunksize=1_000_000):
    ch.columns = [c.strip() for c in ch.columns]
    ch["Timestamp"] = ch["Timestamp"].astype("int64")
    m = False
    for lo, hi in bounds:
        m = m | ((ch["Timestamp"] >= lo) & (ch["Timestamp"] < hi))
    if m.any():
        parts.append(ch[m])
if not parts:
    sys.exit("ПОМИЛКА: у CSV немає жодного рядка для вказаних періодів " + a.periods)
df = pd.concat(parts).sort_values("Timestamp").reset_index(drop=True)
if a.max_rows:
    df = df.head(a.max_rows)
print(f"[producer] рядків до відправки: {len(df)}; затримка {a.delay_ms} мс; тема {a.topic}", flush=True)

prod = Producer({"bootstrap.servers": a.bootstrap, "linger.ms": 5, "acks": "1"})
errors = []
def on_delivery(err, msg):  # зворотний виклик: рахуємо помилки доставки
    if err is not None:
        errors.append(str(err))

def num(x):  # NaN -> null у JSON
    return None if pd.isna(x) else float(x)

t0 = time.time()
sent = 0
for row in df.itertuples(index=False):
    ts = int(row.Timestamp)
    key = time.strftime("%Y-%m-%d", time.gmtime(ts))
    val = {"ts": ts, "open": num(row.Open), "high": num(row.High), "low": num(row.Low),
           "close": num(row.Close), "volume": num(row.Volume)}
    prod.produce(a.topic, key=key, value=json.dumps(val), on_delivery=on_delivery)
    prod.poll(0)
    sent += 1
    if sent % 2000 == 0:
        print(f"[producer] надіслано {sent}/{len(df)}  ({sent/(time.time()-t0):.1f} повід./с)", flush=True)
    time.sleep(a.delay_ms / 1000.0)
prod.flush(60)
el = time.time() - t0
stats = {"sent": sent, "delivery_errors": len(errors), "seconds": round(el, 3),
         "msg_per_s": round(sent / el, 3), "delay_ms": a.delay_ms, "topic": a.topic, "periods": a.periods}
print("[producer] завершено:", json.dumps(stats, ensure_ascii=False), flush=True)
if a.stats:
    json.dump(stats, open(a.stats, "w"))
if a.done_flag:
    json.dump(stats, open(a.done_flag, "w"))
sys.exit(1 if errors else 0)

In [ ]:
# --- smoke-тест producer на окремому topic + console consumer (доказ отримання повідомлень)
with timed("Завдання 2: smoke-тест producer і console consumer"):
    sh(f"{KT} --create --topic bigdata-smoke --partitions 1 --replication-factor 1 --if-not-exists", quiet=True)
    rc,out = sh(f"python3 {CODE}/producer.py --csv '{CSV}' --topic bigdata-smoke --max-rows 30 --delay-ms 20 --periods '{PERIODS}'",
                shot="05_producer_smoke", log="producer_smoke.log", timeout=300)
    check("Завдання 2: producer відправляє повідомлення", rc==0 and "завершено" in out, "smoke-тест 30 повідомлень")
    cc = (f"{KAFKA}/bin/kafka-console-consumer.sh --bootstrap-server {BOOT} --topic bigdata-smoke --from-beginning "
          f"--max-messages 6 --timeout-ms 20000 --group pz3-demo --property print.key=true --property print.timestamp=true")
    rc,out = sh(cc, shot="06_console_consumer_smoke", log="console_consumer_smoke.log", timeout=60)
    check("Завдання 2: console consumer отримав повідомлення", out.count('"ts"')>=6, "bigdata-smoke")
    sh(f"{KAFKA}/bin/kafka-consumer-groups.sh --bootstrap-server {BOOT} --describe --group pz3-demo", shot="07_consumer_group_describe")

## 4. Завдання 3–6. Spark Structured Streaming-застосунок
`streaming_job.py` виконує: **(3)** читання з Kafka (`readStream`, `maxOffsetsPerTrigger`); **(4)** декодування (bytes→string→JSON з явною схемою), перетворення типів (`Long`/`Double`, `unix → timestamp`), фільтрацію некоректних свічок (null, `high<low`, `close<=0`, `volume<0`), групування за добою та статистичні агрегати; **(5)** tumbling-вікна 15 хв і 1 год за **часом події** (`ts`), а не часом обробки: OHLC (`open` – значення з найранішою подією у вікні, `close` – з найпізнішою через `min/max(struct(event_time, …))`), `sum(volume)`, `count`; **(6)** правило аномалій у `foreachBatch` зі станом (буфер 61 хв у pandas); **(7)** parquet-приймачі з checkpoint (append), memory-приймачі (update) для перевірки, console-приймач для демонстрації.

**Watermark = 10 хв.** Дані з одного partition приходять упорядковано, тож запізнілих подій немає; 10 хв допускають дрібні розбіжності (повторна доставка, зміщення між кількома producer) і менші за найкоротше вікно (15 хв). Ціна: parquet-приймач (append) випускає вікно лише коли watermark перетне його кінець, тобто із запізненням на 10 хв події-часу. Тому останні вікна останнього періоду в parquet не потрапляють; їх повний стан береться з memory-приймача (update).

**Вибір реалізації аномалій.** Ковзна медіана/MAD не є алгебраїчним агрегатом і погано ложиться на віконні агрегати Structured Streaming, тому використано `foreachBatch` зі станом у драйвері (буфер останніх 61 записів). Та сама функція `score()` застосовується офлайн (перевірка збігу у §7). Правило: `|r_t| > k·1,4826·MAD_60(r)` **АБО** `V_t > k_v·median_60(V)`, де `r_t = ln(C_t/C_{t-1})`, вікно – 60 попередніх хвилин без поточної.

In [ ]:
%%writefile /content/pz3/code/streaming_job.py
# -*- coding: utf-8 -*-
"""Spark Structured Streaming: Kafka -> розбір JSON -> віконна агрегація -> аномалії -> parquet (ПЗ3, завдання 3-7)."""
import argparse, glob, json, os, sys, time
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession, functions as F, types as T

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from anomaly_rule import StreamScorer, K_GRID

ap = argparse.ArgumentParser()
ap.add_argument("--bootstrap", default="localhost:9092")
ap.add_argument("--topic", default="bigdata-topic")
ap.add_argument("--out", required=True)
ap.add_argument("--ckpt", required=True)
ap.add_argument("--done-flag", required=True)
ap.add_argument("--kafka-pkg", required=True)
ap.add_argument("--watermark", default="10 minutes")
ap.add_argument("--trigger-seconds", type=int, default=10)
ap.add_argument("--max-offsets", type=int, default=3000)
ap.add_argument("--k-stream", type=int, default=5)
ap.add_argument("--max-seconds", type=int, default=1200)
a = ap.parse_args()

os.makedirs(a.out, exist_ok=True)
PARTS = os.path.join(a.out, "scored_parts"); os.makedirs(PARTS, exist_ok=True)

spark = (SparkSession.builder.appName("PZ3-bitcoin-streaming").master("local[2]")
         .config("spark.jars.packages", a.kafka_pkg)
         .config("spark.sql.shuffle.partitions", "4")
         .config("spark.sql.session.timeZone", "UTC")     # вікна вирівнюються по UTC
         .config("spark.driver.memory", "2g")
         .config("spark.ui.port", "4040")
         .config("spark.sql.streaming.forceDeleteTempCheckpointLocation", "true")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")
print("[job] Spark", spark.version, "| пакет", a.kafka_pkg, flush=True)

# ---------- Завдання 3: читання з Kafka ----------
raw = (spark.readStream.format("kafka")
       .option("kafka.bootstrap.servers", a.bootstrap)
       .option("subscribe", a.topic)
       .option("startingOffsets", "earliest")
       .option("maxOffsetsPerTrigger", a.max_offsets)   # обмеження розміру мікробатчу
       .option("failOnDataLoss", "false")
       .load())

# ---------- Завдання 4: декодування, типи, фільтрація ----------
schema = T.StructType([                       # явна схема JSON-повідомлення
    T.StructField("ts", T.LongType()), T.StructField("open", T.DoubleType()),
    T.StructField("high", T.DoubleType()), T.StructField("low", T.DoubleType()),
    T.StructField("close", T.DoubleType()), T.StructField("volume", T.DoubleType())])
parsed = (raw.select(F.col("key").cast("string").alias("key"),                # bytes -> string
                     F.col("value").cast("string").alias("json"),
                     F.col("timestamp").cast("double").alias("kafka_epoch"),  # час запису в Kafka (CreateTime)
                     F.col("offset").alias("kafka_offset"))
          .withColumn("m", F.from_json("json", schema))                       # string -> struct
          .select("key", "kafka_epoch", "kafka_offset", "m.*")
          .withColumn("event_time", F.col("ts").cast("timestamp"))            # ЧАС ПОДІЇ (unix с), а не час обробки
          .withColumn("valid", F.col("ts").isNotNull() & F.col("open").isNotNull() & F.col("close").isNotNull()
                      & F.col("high").isNotNull() & F.col("low").isNotNull() & F.col("volume").isNotNull()
                      & (F.col("high") >= F.col("low")) & (F.col("close") > 0) & (F.col("volume") >= 0)))
valid = parsed.filter("valid")                                                # фільтрація некоректних записів

# watermark: допустиме запізнення подій 10 хв (докладніше у ноутбуці)
events = valid.withWatermark("event_time", a.watermark)

# ---------- Завдання 5: віконні агрегації (tumbling) ----------
def win_agg(df, dur):
    g = df.groupBy(F.window("event_time", dur)).agg(
        F.min(F.struct("event_time", "open")).getField("open").alias("open"),    # open = значення з найранішим event_time
        F.max("high").alias("high"), F.min("low").alias("low"),
        F.max(F.struct("event_time", "close")).getField("close").alias("close"), # close = значення з найпізнішим event_time
        F.sum("volume").alias("volume"), F.count(F.lit(1)).alias("n_candles"))
    return g.select(F.col("window.start").alias("win_start"), F.col("window.end").alias("win_end"),
                    "open", "high", "low", "close", "volume", "n_candles")

w15, w1h = win_agg(events, "15 minutes"), win_agg(events, "1 hour")
# групування + статистичне агрегування по календарній добі (без вікна)
daily = (events.groupBy(F.to_date("event_time").alias("trade_date"))
         .agg(F.count(F.lit(1)).alias("n"), F.avg("close").alias("avg_close"), F.stddev("close").alias("std_close"),
              F.min("low").alias("min_low"), F.max("high").alias("max_high"), F.sum("volume").alias("volume")))

trig = {"processingTime": f"{a.trigger_seconds} seconds"}
Q = {}
# ---------- Завдання 7: parquet-приймачі (append: вікно записується після закриття водяною позначкою) ----------
for name, df in (("win15m", w15), ("win1h", w1h)):
    Q[f"sink_{name}"] = (df.writeStream.queryName(f"sink_{name}").outputMode("append").format("parquet")
                         .option("path", os.path.join(a.out, "parquet", name))
                         .option("checkpointLocation", os.path.join(a.ckpt, name)).trigger(**trig).start())
# memory-приймачі (update): повний поточний стан вікон для перевірки та скріншотів
Q["mem_win15m"] = w15.writeStream.queryName("win15m_mem").outputMode("update").format("memory").trigger(**trig).start()
Q["mem_win1h"] = w1h.writeStream.queryName("win1h_mem").outputMode("update").format("memory").trigger(**trig).start()
Q["mem_daily"] = daily.writeStream.queryName("daily_mem").outputMode("update").format("memory").trigger(**trig).start()
# console-приймач для демонстрації (вивід у лог)
Q["console_1h"] = (w1h.writeStream.queryName("console_1h").outputMode("update").format("console")
                   .option("truncate", "false").option("numRows", 6).trigger(**trig).start())

# ---------- Завдання 6: аномалії у foreachBatch зі станом (буфер останніх 61 хвилини) ----------
scorer = StreamScorer()
BATCH_LOG = []
def process_batch(bdf, batch_id):
    t0 = time.time()
    pdf = bdf.select("ts", "close", "volume", "valid", "kafka_epoch").toPandas()
    n_raw, n_valid = len(pdf), int(pdf["valid"].sum())
    lat = (time.time() - pdf["kafka_epoch"]).to_numpy() if n_raw else np.array([np.nan])
    res = scorer.update(pdf[pdf["valid"]]) if n_valid else pd.DataFrame()
    n_anom = 0
    if len(res):
        res = res.copy()
        res["batch_id"] = batch_id
        res.to_parquet(os.path.join(PARTS, f"part-{batch_id:05d}.parquet"), index=False)  # ідемпотентний запис
        n_anom = int(res[f"flag_k{a.k_stream}"].sum())
    rec = {"batch_id": batch_id, "n_raw": n_raw, "n_invalid": n_raw - n_valid, "n_valid": n_valid,
           "n_anomalies_k%d" % a.k_stream: n_anom, "latency_mean_s": float(np.nanmean(lat)),
           "latency_p95_s": float(np.nanpercentile(lat, 95)), "latency_max_s": float(np.nanmax(lat)),
           "foreachbatch_s": round(time.time() - t0, 3)}
    BATCH_LOG.append(rec)
    print(f"[anomaly] batch {batch_id}: рядків {n_raw}, аномалій(k={a.k_stream}) {n_anom}, "
          f"затримка середня {rec['latency_mean_s']:.1f} с", flush=True)

Q["anomaly"] = (parsed.writeStream.queryName("anomaly").foreachBatch(process_batch)
                .option("checkpointLocation", os.path.join(a.ckpt, "anomaly")).trigger(**trig).start())
print("STREAMING_STARTED", [q.name for q in Q.values()], flush=True)

# ---------- моніторинг: збір прогресу, умова зупинки ----------
PROG = {}
def collect():
    for key, q in Q.items():
        for p in q.recentProgress:
            PROG[(key, p["batchId"])] = {
                "query": key, "batch_id": p["batchId"], "timestamp": p["timestamp"],
                "num_input_rows": p["numInputRows"], "input_rows_per_s": p.get("inputRowsPerSecond"),
                "processed_rows_per_s": p.get("processedRowsPerSecond"),
                "trigger_execution_ms": p["durationMs"].get("triggerExecution"),
                "add_batch_ms": p["durationMs"].get("addBatch"),
                "get_batch_ms": p["durationMs"].get("getBatch"),
                "watermark": (p.get("eventTime") or {}).get("watermark")}
def totals():
    t = {k: 0 for k in Q}
    for (k, _), v in PROG.items():
        t[k] += v["num_input_rows"] or 0
    return t

t_start, status, expected = time.time(), "timeout", None
while time.time() - t_start < a.max_seconds:
    time.sleep(5)
    collect()
    for k, q in Q.items():
        if q.exception():
            print("ПОМИЛКА у запиті", k, q.exception(), flush=True); status = "query_failed"
    if status == "query_failed":
        break
    if expected is None and os.path.exists(a.done_flag):
        try:
            expected = json.load(open(a.done_flag))["sent"]
        except Exception:
            expected = None
    tt = totals()
    print(f"[job] t={time.time()-t_start:.0f}с прочитано(anomaly)={tt['anomaly']} очікується={expected}", flush=True)
    if expected is not None and all(v >= expected for v in tt.values()):
        time.sleep(2 * a.trigger_seconds)   # дати завершитися останнім мікробатчам
        collect(); status = "ok"
        break

# ---------- збереження результатів ----------
summary = {"status": status, "expected_messages": expected, "input_totals": totals(),
           "seconds": round(time.time() - t_start, 1)}
if status in ("ok", "timeout"):
    for tbl, fn in (("win15m_mem", "win15m_all"), ("win1h_mem", "win1h_all"), ("daily_mem", "daily_stats")):
        d = spark.table(tbl).toPandas()
        d = d.sort_values(d.columns[0]).reset_index(drop=True)
        d.to_csv(os.path.join(a.out, fn + ".csv"), index=False)
        d.to_parquet(os.path.join(a.out, fn + ".parquet"), index=False)
        summary[fn + "_rows"] = len(d)
        print(f"[job] {tbl}: {len(d)} рядків -> {fn}.csv", flush=True)
        if tbl == "win1h_mem":
            print(d.head(8).to_string(index=False), flush=True)
pd.DataFrame(list(PROG.values())).sort_values(["query", "batch_id"]).to_csv(os.path.join(a.out, "progress.csv"), index=False)
pd.DataFrame(BATCH_LOG).to_csv(os.path.join(a.out, "batch_stats.csv"), index=False)
parts = sorted(glob.glob(os.path.join(PARTS, "part-*.parquet")))
if parts:
    sc = pd.concat([pd.read_parquet(p) for p in parts], ignore_index=True).sort_values("ts").reset_index(drop=True)
    sc["datetime_utc"] = pd.to_datetime(sc["ts"], unit="s", utc=True).dt.strftime("%Y-%m-%d %H:%M:%S")
    sc.to_parquet(os.path.join(a.out, "scored.parquet"), index=False)
    sc.to_csv(os.path.join(a.out, "scored.csv"), index=False)
    an = sc[sc[f"flag_k{a.k_stream}"]]
    an.to_parquet(os.path.join(a.out, "anomalies.parquet"), index=False)
    an.to_csv(os.path.join(a.out, "anomalies.csv"), index=False)
    summary.update(scored_rows=len(sc), anomalies_rows=len(an), k_stream=a.k_stream)
    print(f"[job] scored={len(sc)} anomalies(k={a.k_stream})={len(an)}", flush=True)
json.dump(summary, open(os.path.join(a.out, "job_summary.json"), "w"), indent=1)
for q in Q.values():
    try: q.stop()
    except Exception: pass
spark.stop()
print("JOB_DONE", json.dumps(summary), flush=True)
sys.exit(0 if status == "ok" else 3)

In [ ]:
%%writefile /content/pz3/code/ui_shot.py
# -*- coding: utf-8 -*-
"""Знімок Spark UI через headless Chromium (playwright). Використання: ui_shot.py URL OUT.png"""
import sys
from playwright.sync_api import sync_playwright
url, out = sys.argv[1], sys.argv[2]
with sync_playwright() as p:
    b = p.chromium.launch(args=["--no-sandbox"])
    pg = b.new_page(viewport={"width": 1400, "height": 900})
    pg.goto(url, wait_until="networkidle", timeout=45000)
    pg.screenshot(path=out, full_page=True)
    b.close()
print("saved", out)

## 5. Запуск потоку (завдання 2–7 разом)
Спочатку стартує Spark-застосунок (чекає на дані), потім producer. Під час роботи знімаються: console consumer на `bigdata-topic`, зміщення (offsets) у Kafka, Spark UI (Jobs, Structured Streaming).

In [ ]:
def tail(path, n=25): return sh(f"tail -n {n} '{path}'", quiet=True)[1]
with timed("Потокова обробка (Spark + producer)"):
    sh(f"rm -rf {CKPT} {JOB_OUT} {FLAG}", quiet=True)
    jenv = os.environ.copy()
    jlog = open(f"{LOGS}/streaming_job.log","w")
    JOB = subprocess.Popen([sys.executable, f"{CODE}/streaming_job.py", "--bootstrap", BOOT, "--topic", TOPIC,
        "--out", JOB_OUT, "--ckpt", CKPT, "--done-flag", FLAG, "--kafka-pkg", KAFKA_PKG, "--watermark", WATERMARK,
        "--trigger-seconds", str(TRIGGER_SEC), "--k-stream", str(K_STREAM)], stdout=jlog, stderr=subprocess.STDOUT, env=jenv)
    started=False
    for i in range(120):   # до 10 хв (завантаження jar-пакетів Maven при першому запуску)
        time.sleep(5)
        if JOB.poll() is not None: break
        if "STREAMING_STARTED" in open(f"{LOGS}/streaming_job.log",errors="ignore").read(): started=True; break
    require(started, "Spark-застосунок не запустив потоки. Кінець логу:\n"+tail(f"{LOGS}/streaming_job.log", 40))
    print("Spark-запити запущено.")
    plog = open(f"{LOGS}/producer.log","w")
    PROD = subprocess.Popen([sys.executable, f"{CODE}/producer.py", "--csv", CSV, "--topic", TOPIC, "--periods", PERIODS,
        "--delay-ms", str(DELAY_MS), "--stats", f"{RES}/producer_stats.json", "--done-flag", FLAG], stdout=plog, stderr=subprocess.STDOUT)
    t0=time.time(); shot_done=set()
    while True:
        time.sleep(15); el=time.time()-t0
        lp=tail(f"{LOGS}/producer.log",1).strip(); lj=[l for l in tail(f"{LOGS}/streaming_job.log",40).splitlines() if l.startswith("[job]") or l.startswith("[anomaly]")]
        print(f"t={el:4.0f}с | {lp} | {lj[-1] if lj else ''}")
        if el>=45 and "kafka" not in shot_done:
            shot_done.add("kafka")
            sh(f"{KAFKA}/bin/kafka-console-consumer.sh --bootstrap-server {BOOT} --topic {TOPIC} --from-beginning --max-messages 8 "
               f"--timeout-ms 15000 --property print.key=true --property print.timestamp=true", shot="08_console_consumer_main", log="console_consumer_main.log", timeout=60)
            sh(f"{KAFKA}/bin/kafka-get-offsets.sh --bootstrap-server {BOOT} --topic {TOPIC} --time -1 || {KAFKA}/bin/kafka-run-class.sh kafka.tools.GetOffsetShell --bootstrap-server {BOOT} --topic {TOPIC}",
               shot="09_kafka_end_offsets")
        if el>=75 and "ui" not in shot_done and UI_OK:
            shot_done.add("ui")
            for url,nm in (("jobs/","10_spark_ui_jobs"),("StreamingQuery/","11_spark_ui_streaming")):
                try:
                    rc,o = sh(f"python3 {CODE}/ui_shot.py http://localhost:4040/{url} {SHOTS}/{nm}.png", quiet=True, timeout=90)
                    print("Spark UI", url, "->", "OK" if rc==0 else "не вдалося: "+o[-200:])
                    if rc==0: SHOT_LIST.append(nm)
                except Exception as ex: print("Spark UI знімок пропущено:", ex)
        if PROD.poll() is not None and JOB.poll() is not None: break
        if el>1500:
            PROD.kill(); JOB.kill(); break
    print("producer rc =", PROD.returncode, "| job rc =", JOB.returncode)
    jlog.close(); plog.close()
    print(tail(f"{LOGS}/producer.log",3)); print(tail(f"{LOGS}/streaming_job.log",12))
    require(PROD.returncode==0, "producer завершився з помилкою:\n"+tail(f"{LOGS}/producer.log",15))
    require(os.path.exists(f"{JOB_OUT}/job_summary.json"), "Spark-застосунок не зберіг job_summary.json. Кінець логу:\n"+tail(f"{LOGS}/streaming_job.log",40))
JSUM = json.load(open(f"{JOB_OUT}/job_summary.json")); PSTAT = json.load(open(f"{RES}/producer_stats.json"))
print(JSUM); sync_drive()
check("Завдання 2: потік відтворено повністю", PSTAT["sent"]==EXPECTED_TOTAL and PSTAT["delivery_errors"]==0,
      f"надіслано {PSTAT['sent']} з {EXPECTED_TOTAL} рядків CSV, {PSTAT['msg_per_s']} повід./с")
check("Завдання 3: Spark прочитав усі повідомлення з Kafka", JSUM["status"]=="ok" and all(v>=PSTAT["sent"] for v in JSUM["input_totals"].values()),
      f"прочитано {JSUM['input_totals']}")

In [ ]:
# --- докази: вивід консольного приймача, структура sink-каталогів і checkpoint
log = open(f"{LOGS}/streaming_job.log",errors="ignore").read()
bids = sorted({int(x) for x in re.findall(r"^Batch: (\d+)$", log, flags=re.M)})
print("пакети console-приймача:", bids[:10], "...")
if bids:
    b = bids[len(bids)//2]
    sh(f"grep -A 14 -m1 '^Batch: {b}$' {LOGS}/streaming_job.log", shot="12_console_sink_batch")
sh(f"grep '^\\[anomaly\\]' {LOGS}/streaming_job.log | head -n 25", shot="13_foreachbatch_anomaly_log")
sh(f"cd {BASE} && ls -la job_out job_out/parquet/win15m | head -n 30 && echo && ls ckpt && echo && find ckpt/win15m -maxdepth 2 | head -n 12", shot="14_sinks_and_checkpoint")

## 6. Завдання 5 і 7. Перевірка віконних агрегацій та збережених результатів

In [ ]:
with timed("Перевірка вікон проти pandas"):
    # 1) parquet-приймач (append) не порожній
    w15_sink = pd.read_parquet(f"{JOB_OUT}/parquet/win15m"); w1h_sink = pd.read_parquet(f"{JOB_OUT}/parquet/win1h")
    check("Завдання 7: parquet-приймач вікон непорожній", len(w15_sink)>0 and len(w1h_sink)>0, f"15 хв: {len(w15_sink)} вікон, 1 год: {len(w1h_sink)} вікон")
    require(len(w15_sink)>0, "parquet-приймач порожній: жодне вікно не закрилося watermark-ом")
    # 2) еталон: pandas.resample по тих самих рядках CSV
    lo_hi=[]
    for it in PERIODS.split(","):
        s,e=it.split(":"); lo_hi.append((int(pd.Timestamp(s,tz="UTC").timestamp()), int((pd.Timestamp(e,tz="UTC")+pd.Timedelta(days=1)).timestamp())))
    parts=[]
    for ch in pd.read_csv(CSV, chunksize=1_000_000):
        m=False
        for lo,hi in lo_hi: m = m | ((ch["Timestamp"]>=lo)&(ch["Timestamp"]<hi))
        if m.any(): parts.append(ch[m])
    ref = pd.concat(parts).dropna(); ref = ref[(ref.High>=ref.Low)&(ref.Close>0)&(ref.Volume>=0)]
    ref["dt"]=pd.to_datetime(ref.Timestamp.astype("int64"),unit="s")
    ref=ref.sort_values("dt").set_index("dt")
    def ref_win(freq):
        r=ref.resample(freq).agg({"Open":"first","High":"max","Low":"min","Close":"last","Volume":"sum","Timestamp":"count"})
        r=r[r.Timestamp>0]; r.columns=["open","high","low","close","volume","n_candles"]; r.index.name="win_start"; return r.reset_index()
    def compare(spark_df, ref_df, label):
        s=spark_df.copy(); s["win_start"]=pd.to_datetime(s["win_start"]).dt.tz_localize(None) if pd.to_datetime(s["win_start"]).dt.tz is not None else pd.to_datetime(s["win_start"])
        m=s.merge(ref_df,on="win_start",how="outer",suffixes=("_spark","_ref"),indicator=True)
        both=m[m._merge=="both"]
        bad=0
        for c in ("open","high","low","close","volume"):
            bad+=int((~np.isclose(both[c+"_spark"],both[c+"_ref"],rtol=1e-9,atol=1e-9)).sum())
        bad+=int((both["n_candles_spark"]!=both["n_candles_ref"]).sum())
        return {"window":label,"spark_windows":len(s),"reference_windows":len(ref_df),"only_spark":int((m._merge=="left_only").sum()),
                "only_reference":int((m._merge=="right_only").sum()),"value_mismatches":bad}
    w15_all=pd.read_csv(f"{JOB_OUT}/win15m_all.csv"); w1h_all=pd.read_csv(f"{JOB_OUT}/win1h_all.csv")
    cmpdf=pd.DataFrame([compare(w15_all,ref_win("15min"),"15 хв (memory, update)"), compare(w1h_all,ref_win("1h"),"1 год (memory, update)"),
                        compare(w15_sink,ref_win("15min"),"15 хв (parquet, append)"), compare(w1h_sink,ref_win("1h"),"1 год (parquet, append)")])
    cmpdf.to_csv(f"{RES}/window_check_vs_pandas.csv",index=False); print(cmpdf.to_string(index=False))
    # у append-приймачі відсутні лише останні вікна (watermark ще не перетнув їх кінець)
    full_ok = (cmpdf.loc[:1,"only_spark"].sum()==0 and cmpdf.loc[:1,"only_reference"].sum()==0 and cmpdf.loc[:1,"value_mismatches"].sum()==0)
    sink_ok = (cmpdf.loc[2:,"only_spark"].sum()==0 and cmpdf.loc[2:,"value_mismatches"].sum()==0)
    check("Завдання 5: OHLC/volume/count вікон збігаються з pandas.resample", full_ok, "memory-приймач, 15 хв і 1 год")
    check("Завдання 7: вікна у parquet збігаються з еталоном", sink_ok, f"не випущено watermark-ом: 15 хв {cmpdf.loc[2,'only_reference']}, 1 год {cmpdf.loc[3,'only_reference']} вікон")
    check("Завдання 4: сума n_candles = кількість коректних повідомлень", int(w15_all.n_candles.sum())==len(ref) and int(w1h_all.n_candles.sum())==len(ref),
          f"{int(w15_all.n_candles.sum())} vs {len(ref)} (запізнілих подій немає)")

In [ ]:
# --- таблиці: агрегати за вікнами, добова статистика, сліди сховища (скріншоти реального виводу)
sh(f"python3 -c \"import pandas as pd; pd.set_option('display.width',200); d=pd.read_parquet('{JOB_OUT}/parquet/win15m').sort_values('win_start'); print(d.head(10).to_string(index=False)); print(len(d),'вікон 15 хв у parquet')\"", shot="15_parquet_win15m")
sh(f"python3 -c \"import pandas as pd; pd.set_option('display.width',200); d=pd.read_parquet('{JOB_OUT}/parquet/win1h').sort_values('win_start'); print(d.head(10).to_string(index=False)); print(len(d),'вікон 1 год у parquet')\"", shot="16_parquet_win1h")
sh(f"cat {JOB_OUT}/daily_stats.csv", shot="17_daily_stats")
sh(f"cat {RES}/window_check_vs_pandas.csv", shot="18_window_check")
sh(f"ls -la {JOB_OUT}/*.csv {JOB_OUT}/*.parquet", shot="19_saved_files")

In [ ]:
# --- пропускна здатність, тривалість мікробатчів, затримка
prog=pd.read_csv(f"{JOB_OUT}/progress.csv"); bst=pd.read_csv(f"{JOB_OUT}/batch_stats.csv")
prog.to_csv(f"{RES}/batch_durations_all_queries.csv",index=False); bst.to_csv(f"{RES}/latency_per_batch.csv",index=False)
a_=prog[(prog["query"]=="anomaly")&(prog.num_input_rows>0)]
thr=pd.DataFrame([{
 "producer_msg_per_s":PSTAT["msg_per_s"],"producer_seconds":PSTAT["seconds"],"messages":PSTAT["sent"],"delay_ms":PSTAT["delay_ms"],
 "spark_input_rows_per_s_mean":a_.input_rows_per_s.mean(),"spark_processed_rows_per_s_mean":a_.processed_rows_per_s.mean(),
 "spark_processed_rows_per_s_max":a_.processed_rows_per_s.max(),"microbatches_nonempty":len(a_),
 "trigger_ms_mean":a_.trigger_execution_ms.mean(),"trigger_ms_p95":a_.trigger_execution_ms.quantile(.95),"trigger_ms_max":a_.trigger_execution_ms.max(),
 "latency_mean_s":np.average(bst.latency_mean_s,weights=bst.n_raw),"latency_p95_s_max_over_batches":bst.latency_p95_s.max(),"latency_max_s":bst.latency_max_s.max(),
 "invalid_records":int(bst.n_invalid.sum()),"raw_records":int(bst.n_raw.sum())}]).round(3)
thr.to_csv(f"{RES}/throughput_latency.csv",index=False)
sh(f"python3 -c \"import pandas as pd; print(pd.read_csv('{RES}/throughput_latency.csv').T.to_string(header=False))\"", shot="20_throughput_latency")
check("Завдання 4: відфільтровано некоректні записи (лічильник)", True, f"некоректних: {int(bst.n_invalid.sum())} із {int(bst.n_raw.sum())}")
print(write_summary()[-400:])

## 7. Завдання 6. Калібрування правила аномалій на всій історії
`anomaly_analysis.py` застосовує ту саму функцію `score()` до **всієї історії** (pandas/numpy), рахує кількість спрацювань для k ∈ {3, 5, 7, 10} (k_v = k), обирає k за «бюджетом» (частка позначених хвилин ≤ 1 %, тобто не більше ~14 сигналів на добу), показує реальні дати з найбільшою кількістю аномалій і звіряє результат потокової реалізації з офлайн-реалізацією на тих самих хвилинах.

In [ ]:
%%writefile /content/pz3/code/anomaly_analysis.py
# -*- coding: utf-8 -*-
"""Офлайн-калібрування правила аномалій на всій історії та аналіз результатів потоку (ПЗ3, завдання 6 і 8)."""
import argparse, json, os, sys, time
import numpy as np
import pandas as pd

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
from anomaly_rule import score, K_GRID

ap = argparse.ArgumentParser()
ap.add_argument("--csv", required=True)
ap.add_argument("--out", required=True)
ap.add_argument("--scored", default="", help="scored.parquet з потокового застосунку")
ap.add_argument("--budget", type=float, default=0.01, help="допустима частка позначених хвилин")
a = ap.parse_args()
os.makedirs(a.out, exist_ok=True)
fig_dir = os.path.join(a.out, "figures"); os.makedirs(fig_dir, exist_ok=True)

# --- 1. завантаження повної історії
t0 = time.time()
df = pd.read_csv(a.csv, usecols=["Timestamp", "Close", "Volume"])
df.columns = ["ts", "close", "volume"]
n_all = len(df)
df = df.dropna()
df["ts"] = df["ts"].astype("int64")
df = df[(df["close"] > 0) & (df["volume"] >= 0)]
print(f"[analysis] рядків у CSV: {n_all}; коректних: {len(df)}; читання {time.time()-t0:.0f} с", flush=True)

# --- 2. оцінка правила на всій історії
t0 = time.time()
sc = score(df)
sc["date"] = pd.to_datetime(sc["ts"], unit="s", utc=True).dt.strftime("%Y-%m-%d")
print(f"[analysis] score() на {len(sc)} рядках: {time.time()-t0:.0f} с", flush=True)
v = sc[sc["scored"]]
nv = len(v)
rows = []
for k in K_GRID:
    n_ret, n_vol = int((v["z_ret"] > k).sum()), int((v["vol_ratio"] > k).sum())
    n_u = int(v[f"flag_k{k}"].sum())
    rows.append({"k": k, "n_valid_minutes": nv, "n_return_rule": n_ret, "n_volume_rule": n_vol, "n_union": n_u,
                 "share_union_pct": round(100 * n_u / nv, 4), "per_day_avg": round(n_u / (nv / 1440), 3)})
cnt = pd.DataFrame(rows)
cnt.to_csv(os.path.join(a.out, "anomaly_counts_per_k.csv"), index=False)
print(cnt.to_string(index=False), flush=True)
ok = cnt[cnt["share_union_pct"] <= 100 * a.budget]
k_sel = int(ok["k"].min()) if len(ok) else int(cnt["k"].max())
reason = (f"найменше k із сітки, при якому частка позначених хвилин <= {100*a.budget:.1f}%" if len(ok)
          else f"жодне k не вкладається у бюджет {100*a.budget:.1f}%, взято найбільше")
print(f"[analysis] обрано k = k_v = {k_sel} ({reason})", flush=True)
json.dump({"k": k_sel, "reason": reason, "budget": a.budget}, open(os.path.join(a.out, "chosen_k.json"), "w"))

# --- 3. на які реальні дати припадають аномалії
fl = v[v[f"flag_k{k_sel}"]].copy()
fl["score"] = np.fmax(fl["z_ret"].fillna(0) / k_sel, fl["vol_ratio"].fillna(0) / k_sel)
by_date = (fl.groupby("date").agg(n_anomalies=("ts", "size"), max_z_ret=("z_ret", "max"), max_vol_ratio=("vol_ratio", "max"),
                                  max_abs_r=("r", lambda s: s.abs().max())).sort_values("n_anomalies", ascending=False))
by_date.head(30).to_csv(os.path.join(a.out, "top_dates.csv"))
print("[analysis] топ-15 дат за кількістю аномалій:\n", by_date.head(15).to_string(), flush=True)
top_min = fl.sort_values("score", ascending=False).head(30).copy()
top_min["datetime_utc"] = pd.to_datetime(top_min["ts"], unit="s", utc=True).dt.strftime("%Y-%m-%d %H:%M")
top_min[["datetime_utc", "close", "volume", "r", "z_ret", "vol_ratio", "score"]].to_csv(
    os.path.join(a.out, "top_minutes.csv"), index=False)
fl["year"] = fl["date"].str[:4]
fl.groupby("year").size().rename("n_anomalies").to_csv(os.path.join(a.out, "anomalies_by_year.csv"))
chk = []
for d in ("2020-03-12", "2020-03-13", "2021-05-19", "2022-11-08", "2022-11-09"):   # перевірка відомих подій
    rank = int(by_date.index.get_loc(d)) + 1 if d in by_date.index else None
    chk.append({"date": d, "n_anomalies": int(by_date.loc[d, "n_anomalies"]) if d in by_date.index else 0, "rank_by_count": rank})
pd.DataFrame(chk).to_csv(os.path.join(a.out, "known_events_check.csv"), index=False)
print(pd.DataFrame(chk).to_string(index=False), flush=True)

# --- 4. порівняння потокової та офлайн-реалізацій на тих самих хвилинах
if a.scored and os.path.exists(a.scored):
    st = pd.read_parquet(a.scored)
    sub = score(df[df["ts"].isin(st["ts"])])
    m = st.merge(sub, on="ts", suffixes=("_stream", "_off"))
    both = m[m["scored_stream"] & m["scored_off"]]
    cmp_ = {"n_stream": len(st), "n_both_scored": len(both),
            "max_abs_diff_z": float(np.nanmax(np.abs(both["z_ret_stream"] - both["z_ret_off"]))) if len(both) else None,
            "max_abs_diff_q": float(np.nanmax(np.abs(both["vol_ratio_stream"] - both["vol_ratio_off"]))) if len(both) else None,
            "flag_mismatch_k%d" % k_sel: int((m[f"flag_k{k_sel}_stream"] != m[f"flag_k{k_sel}_off"]).sum())}
    pd.DataFrame([cmp_]).to_csv(os.path.join(a.out, "stream_vs_offline.csv"), index=False)
    print("[analysis] потік проти офлайн:", cmp_, flush=True)
    s2 = st[st["scored"]].copy()
    an = s2[s2[f"flag_k{k_sel}"]].copy()
    an["datetime_utc"] = pd.to_datetime(an["ts"], unit="s", utc=True).dt.strftime("%Y-%m-%d %H:%M:%S")
    an["date"] = an["datetime_utc"].str[:10]
    an.to_parquet(os.path.join(a.out, "anomalies_stream_final.parquet"), index=False)
    an.to_csv(os.path.join(a.out, "anomalies_stream_final.csv"), index=False)
    pd.DataFrame({"k": [k_sel], "n_stream_scored": [len(s2)], "n_anomalies": [len(an)],
                  "dates": [";".join(sorted(an["date"].unique()))]}).to_csv(os.path.join(a.out, "stream_anomaly_summary.csv"), index=False)
    print(f"[analysis] у відтвореному потоці при k={k_sel}: {len(an)} аномалій із {len(s2)} оцінених хвилин", flush=True)
    print(an.groupby("date").size().rename("n").to_string(), flush=True)

# --- 5. рисунки
try:
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(7, 4))
    x = np.arange(len(cnt)); wdt = 0.27
    ax.bar(x - wdt, cnt["n_return_rule"], wdt, label="правило за дохідністю")
    ax.bar(x, cnt["n_volume_rule"], wdt, label="правило за обсягом")
    ax.bar(x + wdt, cnt["n_union"], wdt, label="об'єднання (АБО)")
    ax.set_xticks(x); ax.set_xticklabels([f"k={k}" for k in cnt["k"]]); ax.set_yscale("log")
    ax.set_ylabel("кількість хвилин"); ax.set_title("Кількість спрацювань залежно від k"); ax.legend()
    fig.tight_layout(); fig.savefig(os.path.join(fig_dir, "anomaly_counts_per_k.png"), dpi=150); plt.close(fig)
    if a.scored and os.path.exists(a.scored):
        st = pd.read_parquet(a.scored); st["dt"] = pd.to_datetime(st["ts"], unit="s", utc=True)
        periods = [g for _, g in st.groupby((st["ts"].diff() > 7200).cumsum())]
        fig, axs = plt.subplots(len(periods), 1, figsize=(9, 2.6 * len(periods)))
        axs = np.atleast_1d(axs)
        for ax_, g in zip(axs, periods):
            ax_.plot(g["dt"], g["close"], lw=0.8)
            an_ = g[g[f"flag_k{k_sel}"]]
            ax_.scatter(an_["dt"], an_["close"], s=8, c="red", label=f"аномалії k={k_sel}")
            ax_.set_ylabel("Close, USD"); ax_.legend(loc="upper left")
        fig.tight_layout(); fig.savefig(os.path.join(fig_dir, "stream_price_anomalies.png"), dpi=150); plt.close(fig)
except Exception as e:
    print("[analysis] рисунки не збережено:", e, flush=True)
print("ANALYSIS_DONE", flush=True)

In [ ]:
with timed("Завдання 6: офлайн-калібрування k на всій історії"):
    os.makedirs(f"{RES}/anomaly",exist_ok=True)
    rc,out = sh(f"python3 {CODE}/anomaly_analysis.py --csv '{CSV}' --out {RES}/anomaly --scored {JOB_OUT}/scored.parquet",
                shot="21_anomaly_analysis", log="anomaly_analysis.log", timeout=1200, quiet=True)
    print(out[-6000:])
    require(rc==0 and "ANALYSIS_DONE" in out, "anomaly_analysis.py завершився з помилкою")
cnt=pd.read_csv(f"{RES}/anomaly/anomaly_counts_per_k.csv"); chosen=json.load(open(f"{RES}/anomaly/chosen_k.json"))
sh(f"cat {RES}/anomaly/anomaly_counts_per_k.csv", shot="22_anomaly_counts_per_k")
sh(f"head -n 16 {RES}/anomaly/top_dates.csv", shot="23_anomaly_top_dates")
sh(f"cat {RES}/anomaly/known_events_check.csv {RES}/anomaly/stream_vs_offline.csv", shot="24_known_events_stream_vs_offline")
svo=pd.read_csv(f"{RES}/anomaly/stream_vs_offline.csv").iloc[0]
check("Завдання 6: правило аномалій реалізовано (потік + офлайн)", len(cnt)==4 and (svo.filter(like="flag_mismatch")==0).all(),
      f"обрано k={chosen['k']}; розбіжностей прапорців потік/офлайн: {int(svo.filter(like='flag_mismatch').iloc[0])}")
an=pd.read_parquet(f"{RES}/anomaly/anomalies_stream_final.parquet")
check("Завдання 7: аномалії збережено (parquet+csv)", len(an)>0 and os.path.exists(f"{RES}/anomaly/anomalies_stream_final.csv"), f"{len(an)} аномалій при k={chosen['k']}")
from IPython.display import display, Image as IPImage
for f in ("anomaly_counts_per_k.png","stream_price_anomalies.png"):
    p=f"{RES}/anomaly/figures/{f}"
    if os.path.exists(p): shutil.copy(p, f"{SHOTS}/fig_{f}"); display(IPImage(p))
sync_drive()

## 8. Завдання 8. Закономірності, виявлені при потоковій обробці

In [ ]:
w1h=pd.read_csv(f"{JOB_OUT}/win1h_all.csv",parse_dates=["win_start"]); w15=pd.read_csv(f"{JOB_OUT}/win15m_all.csv",parse_dates=["win_start"])
daily=pd.read_csv(f"{JOB_OUT}/daily_stats.csv"); sc=pd.read_parquet(f"{JOB_OUT}/scored.parquet")
w1h["range_pct"]=100*(w1h.high-w1h.low)/w1h.open; w15["range_pct"]=100*(w15.high-w15.low)/w15.open
w1h["hour"]=w1h.win_start.dt.hour; w1h["period"]=(w1h.win_start.diff()>pd.Timedelta(days=2)).cumsum()+1
by_hour=w1h.groupby("hour").agg(mean_range_pct=("range_pct","mean"),mean_volume=("volume","mean")).round(4)
by_hour.to_csv(f"{RES}/pattern_by_hour_utc.csv")
daily["range_pct"]=100*(daily.max_high-daily.min_low)/daily.avg_close
per_period=w1h.groupby("period").agg(from_=("win_start","min"),to_=("win_start","max"),mean_range_pct=("range_pct","mean"),max_range_pct=("range_pct","max"),volume=("volume","sum")).round(3)
per_period.to_csv(f"{RES}/pattern_by_period.csv")
daily.sort_values("range_pct",ascending=False).to_csv(f"{RES}/pattern_days_by_range.csv",index=False)
rho=w15[["range_pct","volume"]].corr(method="spearman").iloc[0,1]
k_=chosen["k"]; scs=sc[sc.scored]; scs=scs.assign(hour=pd.to_datetime(scs.ts,unit="s").dt.hour, date=pd.to_datetime(scs.ts,unit="s").dt.strftime("%Y-%m-%d"))
an_by_day=scs[scs[f"flag_k{k_}"]].groupby("date").size().sort_values(ascending=False)
an_by_hour=scs[scs[f"flag_k{k_}"]].groupby("hour").size()
top_day=daily.sort_values("range_pct",ascending=False).iloc[0]
require(len(an_by_day)>0, "у потоці немає жодної аномалії при обраному k – перевірте дані/параметри")
lines=[
 f"1. Волатильність за годинами UTC: максимум середнього діапазону 1-год вікна о {int(by_hour.mean_range_pct.idxmax()):02d}:00 ({by_hour.mean_range_pct.max():.3f} %), мінімум о {int(by_hour.mean_range_pct.idxmin()):02d}:00 ({by_hour.mean_range_pct.min():.3f} %).",
 f"2. Обсяг найбільший о {int(by_hour.mean_volume.idxmax()):02d}:00 UTC (середній обсяг вікна {by_hour.mean_volume.max():.1f} BTC), найменший о {int(by_hour.mean_volume.idxmin()):02d}:00 ({by_hour.mean_volume.min():.1f} BTC).",
 f"3. Кореляція Спірмена між діапазоном і обсягом 15-хв вікон: {rho:.3f}.",
 f"4. Найвищий добовий діапазон (max high – min low відносно середнього close): {top_day.trade_date}, {top_day.range_pct:.2f} %.",
 f"5. Аномалій (k={k_}) у відтвореному потоці: {int(an_by_day.sum())} із {len(scs)} оцінених хвилин ({100*an_by_day.sum()/len(scs):.2f} %); найбільше за датою: {an_by_day.index[0]} ({int(an_by_day.iloc[0])}).",
 f"6. Частка аномалій у три найбільш аномальні години доби: {100*an_by_hour.sort_values(ascending=False).head(3).sum()/an_by_hour.sum():.1f} % (години UTC: {list(an_by_hour.sort_values(ascending=False).head(3).index)}).",
 "7. Середній діапазон 1-год вікна за періодами (таблиця pattern_by_period.csv): "+"; ".join(f"період {i}: {r.mean_range_pct:.3f} %" for i,r in per_period.iterrows())+"."]
open(f"{RES}/findings.txt","w",encoding="utf-8").write("\n".join(lines)+"\n"); print("\n".join(lines))
sh(f"cat {RES}/pattern_by_period.csv && head -n 8 {RES}/pattern_days_by_range.csv", shot="25_patterns")
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
fig,ax=plt.subplots(1,2,figsize=(11,3.8))
ax[0].bar(by_hour.index,by_hour.mean_range_pct); ax[0].set_xlabel("година UTC"); ax[0].set_ylabel("середній діапазон 1-год вікна, %"); ax[0].set_title("Волатильність за годинами")
ax[1].bar(by_hour.index,by_hour.mean_volume,color="tab:orange"); ax[1].set_xlabel("година UTC"); ax[1].set_ylabel("середній обсяг, BTC"); ax[1].set_title("Обсяг за годинами")
fig.tight_layout(); fig.savefig(f"{SHOTS}/fig_pattern_by_hour.png",dpi=150); plt.show()
check("Завдання 8: закономірності обчислено й збережено", True, "findings.txt, pattern_*.csv")

## 9. Підсумок: чекліст, версії, час, збереження на Drive

In [ ]:
TIMES.append(("ВСЬОГО (від початку §0)", round(time.time()-t_global,1)))
for f in glob.glob(f"{JOB_OUT}/parquet"): pass
check("Скріншоти створено", len(glob.glob(f"{SHOTS}/*.png"))>=15, f"{len(glob.glob(f'{SHOTS}/*.png'))} PNG; Spark UI: {'так' if any(s.startswith('10_') for s in SHOT_LIST) else 'не знято (необов’язково)'}")
txt=write_summary(); print(txt)
print("\nУсі файли на Drive:", OUT_DRIVE)
sh(f"find '{OUT_DRIVE}' -maxdepth 2 | head -n 60", quiet=False)
print("Загальний час:", round((time.time()-t_global)/60,1), "хв")